# 01. Data Overview And EDA

This notebook is the project starting point for the denser `10,000` node sample:

- `wash_trading_gnn_nodes_10000.csv`
- `wash_trading_gnn_edges_10000.csv`

It answers four questions before any modeling:

1. What is the class balance?
2. How connected is the sampled graph?
3. Which feature groups are available?
4. Are there missing values or feature issues that need to be handled downstream?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
NODES_PATH = ROOT / "wash_trading_gnn_nodes_10000.csv"
EDGES_PATH = ROOT / "wash_trading_gnn_edges_10000.csv"

nodes_df = pd.read_csv(NODES_PATH)
edges_df = pd.read_csv(EDGES_PATH)

print(f"Nodes path: {NODES_PATH}")
print(f"Edges path: {EDGES_PATH}")


In [ ]:
print("nodes_df.shape =", nodes_df.shape)
print("edges_df.shape =", edges_df.shape)
display(nodes_df.head())
display(edges_df.head())


In [ ]:
feature_groups = {
    "features": [c for c in nodes_df.columns if c.startswith("features_")],
    "normalized_log_features": [c for c in nodes_df.columns if c.startswith("normalized_log_features_")],
    "twitter_semantic_features": [c for c in nodes_df.columns if c.startswith("twitter_semantic_features_")],
    "twitter_semantic_features_normalized": [c for c in nodes_df.columns if c.startswith("twitter_semantic_features_normalized_")],
    "twitter_deepwalk_features": [c for c in nodes_df.columns if c.startswith("twitter_deepwalk_features_")],
    "twitter_deepwalk_normalized_features": [c for c in nodes_df.columns if c.startswith("twitter_deepwalk_normalized_features_")],
    "twitter_combined_features": [c for c in nodes_df.columns if c.startswith("twitter_combined_features_")],
    "eth_twitter_combined_features": [c for c in nodes_df.columns if c.startswith("eth_twitter_combined_features_")],
    "graph_stats": [
        "full_in_degree",
        "full_out_degree",
        "full_total_degree",
        "full_positive_touch_count",
        "full_has_self_loop",
        "sub_in_degree",
        "sub_out_degree",
        "sub_total_degree",
    ],
}

pd.Series({name: len(cols) for name, cols in feature_groups.items()}).sort_values(ascending=False)


In [ ]:
label_counts = nodes_df["label"].value_counts().sort_index()
label_ratio = (label_counts / label_counts.sum()).rename("ratio")
display(pd.concat([label_counts.rename("count"), label_ratio], axis=1))

plt.figure(figsize=(5, 4))
sns.countplot(data=nodes_df, x="label")
plt.title("Label Distribution")
plt.show()


In [ ]:
G = nx.from_pandas_edgelist(
    edges_df,
    source="src_node_id",
    target="dst_node_id",
    create_using=nx.DiGraph(),
)

connected_nodes = set(edges_df["src_node_id"]).union(edges_df["dst_node_id"])
isolated_count = int((~nodes_df["node_id"].isin(connected_nodes)).sum())

graph_summary = {
    "num_nodes_in_table": len(nodes_df),
    "num_edges_in_table": len(edges_df),
    "graph_nodes": G.number_of_nodes(),
    "graph_edges": G.number_of_edges(),
    "isolated_nodes": isolated_count,
    "density": nx.density(G),
    "self_loops": nx.number_of_selfloops(G),
}

pd.Series(graph_summary)


In [ ]:
component_sizes = sorted((len(c) for c in nx.weakly_connected_components(G)), reverse=True)
pd.Series({
    "num_weak_components": len(component_sizes),
    "largest_component_size": component_sizes[0],
    "median_component_size": float(np.median(component_sizes)),
    "top_10_component_sizes": component_sizes[:10],
})


In [ ]:
nan_counts = nodes_df.isna().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)
print(f"Columns with NaNs: {len(nan_counts)}")
display(nan_counts.head(20))


In [ ]:
split_seed = 42

train_ids, temp_ids = train_test_split(
    nodes_df["node_id"],
    test_size=0.30,
    random_state=split_seed,
    stratify=nodes_df["label"],
)

temp_df = nodes_df[nodes_df["node_id"].isin(temp_ids)]
val_ids, test_ids = train_test_split(
    temp_df["node_id"],
    test_size=0.50,
    random_state=split_seed,
    stratify=temp_df["label"],
)

split_summary = []
for split_name, split_ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    part = nodes_df[nodes_df["node_id"].isin(split_ids)]
    split_summary.append(
        {
            "split": split_name,
            "rows": len(part),
            "positives": int(part["label"].sum()),
            "positive_ratio": float(part["label"].mean()),
        }
    )

pd.DataFrame(split_summary)


## Next

- `02_tabular_baselines.ipynb`: feature-only supervised baselines
- `03_graph_feature_baselines.ipynb`: add degree/PageRank/component features
- `04_gnn_models.ipynb`: GraphSAGE, GAT, or GGNN on the sampled graph
